In [1]:
import io

import numpy as np
from pdbfixer import PDBFixer
from openmm.app import PDBFile

import sciapi
import scids
import caddpy
import scishow

In [3]:
r=sciapi.proteinsplus().protoss("1aq1").protein

In [6]:
c=caddpy.chemsys.from_pdb(r)

In [9]:
c.composition.atoms[c.composition.atoms.res_poly]

,chain_id,res_name,res_seq,i_code,res_poly,res_std,serial,name,alt_loc,occupancy,temp_factor,element,charge,element_index
serial,,,,,,,,,,,,,,
1,A,MET,1,,True,True,1,N,,1,0,N,<NA>,6
2,A,MET,1,,True,True,2,CA,,1,0,C,<NA>,5
3,A,MET,1,,True,True,3,C,,1,0,C,<NA>,5
4,A,MET,1,,True,True,4,O,,1,0,O,<NA>,7
5,A,MET,1,,True,True,5,CB,,1,0,C,<NA>,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4515,A,LEU,298,,True,True,4515,HD12,,1,0,H,<NA>,0
4516,A,LEU,298,,True,True,4516,HD13,,1,0,H,<NA>,0
4517,A,LEU,298,,True,True,4517,HD21,,1,0,H,<NA>,0


In [ ]:
PDB_ID = "1AQ1"

In [ ]:
pdb_raw_bytes = sciapi.pdb.file.entry(PDB_ID, "pdb")

In [ ]:
fixer = PDBFixer(pdbfile=io.BytesIO(pdb_raw_bytes))
fixer.findMissingResidues()
missing_residues = fixer.missingResidues
fixer.findNonstandardResidues()
nonstandard_residues = fixer.nonstandardResidues
fixer.replaceNonstandardResidues()
fixer.findMissingAtoms()
missing_atoms = fixer.missingAtoms
fixer.addMissingAtoms()
fixer.addMissingHydrogens(7.0)

pdb_fixed_buffer = io.StringIO()
PDBFile.writeFile(topology=fixer.topology, positions=fixer.positions, file=pdb_fixed_buffer, keepIds=True)
pdb_fixed_buffer.seek(0)
pdb_fixed_str = pdb_fixed_buffer.getvalue()

In [ ]:
pdb_fixed_apo_buffer = io.StringIO()
fixer.removeHeterogens(False)
PDBFile.writeFile(topology=fixer.topology, positions=fixer.positions, file=pdb_fixed_apo_buffer, keepIds=True)
pdb_fixed_apo_buffer.seek(0)
pdb_fixed_apo_str = pdb_fixed_apo_buffer.getvalue()

In [ ]:
rcomplex = caddpy.chemsys.from_pdb(pdb_fixed_str)

In [ ]:
atoms = rcomplex.composition.atoms
ligand_atom_selection = (atoms["res_name"] == "STU").to_numpy()
ligand_atom_coordinates = rcomplex.trajectory.points[ligand_atom_selection]
pocket_center = ligand_atom_coordinates.mean(axis=0)
grid = scids.grid.from_size_spacing_anchor(
    size=(16, 16, 16),
    spacings=0.6,
    anchor="center",
    anchor_coord=pocket_center,
)

In [ ]:
receptor = caddpy.chemsys.from_pdb(pdb_fixed_apo_str)

In [ ]:
energy_field = caddpy.mif.autogrid.from_chemsys(
    system=receptor,
    grid=grid,
    ligand_types=("HD", "OA", "C"),
    include_dsolvmap=False,
)

In [ ]:
hd_mask = energy_field.tensor[0] <= -0.35
oa_mask = energy_field.tensor[1] <= -0.6
c_mask = energy_field.tensor[2] <= -0.4
pi_mask = energy_field.tensor[3] <= -1
ni_mask = energy_field.tensor[3] >= 1

In [ ]:
pocket_field = receptor.toxelate(grid=grid)
ligsite = caddpy.pocket.ligsite.LigSite(field=pocket_field, directions=(1, 3))
pocket_vacancy = energy_field.tensor[1] <= 0.6
pocket_buriedness = ligsite.psp_count >= 4
pocket_mask = np.logical_and(pocket_vacancy, pocket_buriedness)

In [ ]:
is_hd = np.logical_and(hd_mask, pocket_mask)
is_oa = np.logical_and(oa_mask, pocket_mask)
is_c = np.logical_and(c_mask, pocket_mask)
is_pi = np.logical_and(pi_mask, pocket_mask)
is_ni = np.logical_and(ni_mask, pocket_mask)

In [ ]:
hd_points = scids.pointcloud.from_array(grid.coordinates[is_hd])

In [ ]:
hd_clusters = hd_points.cluster_cnn(
    max_distance=1.21,
    min_neighbors=6,
    min_members=5,
    max_members=None
)

In [ ]:
hd_clusters

In [ ]:
np.bincount(hd_clusters)

In [ ]:
viewer = scishow.nglview.NGLWidget()
viewer.add_trajectory(rcomplex)
viewer.add_trajectory(receptor)
viewer.add_spheres(
    coords=grid.coordinates[pocket_vacancy],
    radii=grid.spacings[0]/2,
    name="vacancy",
    representation_params=scishow.nglview.RepresentationParameters(opacity=0.5, visible=True)
)
viewer.add_spheres(
    coords=grid.coordinates[pocket_buriedness],
    radii=grid.spacings[0]/2,
    name="buriedness",
    representation_params=scishow.nglview.RepresentationParameters(opacity=0.5, visible=True)
)
viewer.add_spheres(
    coords=grid.coordinates[pocket_mask],
    radii=grid.spacings[0]/2,
    name="pocket",
    representation_params=scishow.nglview.RepresentationParameters(opacity=0.5, visible=True)
)
viewer.add_spheres(
    coords=grid.coordinates[is_hd],
    radii=grid.spacings[0]/2,
    name="HD",
    representation_params=scishow.nglview.RepresentationParameters(opacity=0.5, visible=True)
)
viewer.add_spheres(
    coords=hd_points[hd_clusters==1],
    radii=grid.spacings[0]/2,
    name="HD Cluster",
    representation_params=scishow.nglview.RepresentationParameters(opacity=0.5, visible=True)
)
viewer.add_box(grid.lower_bounds, grid.upper_bounds)

In [ ]:
grid

In [ ]:
viewer.display(gui=True)

In [ ]:
viewer.add_spheres(
    coords=grid.coordinates[hd_mask],
    radii=grid.spacings[0]/2,
    name="HD Mask",
    representation_params=scishow.nglview.RepresentationParameters(opacity=0.5, visible=True)
)

In [ ]:
import numpy as np
np.count_nonzero(pocket_mask)

In [ ]:
x=np.random.rand(6,5,4,3) > 0.5
x.shape

In [ ]:
y=np.random.rand(5,4,3) > 0.5
y.shape

In [ ]:
np.logical_and(x, y).shape